# RAG (Retrieval-Augmented Generation) Exercise Notebook

**Bonus/Take-Home Exercise** - Build a complete RAG system for movie Q&A

This notebook is a **bonus exercise** that you can complete at home. It demonstrates how to combine search (retrieval) with LLM generation.

**Prerequisites**:
- Complete **Exercise Notebook Part 2** (especially similarity-based search)
- Complete **Exercise Notebook Part 3** (industry tools) - recommended
- Understanding of TF-IDF and cosine similarity
- Basic Python knowledge

**What you'll build**:
- A complete RAG system that retrieves relevant movies and generates answers
- Multiple LLM integration options (mock, Cloudflare, Hugging Face, OpenAI)
- Prompt engineering and context construction

**Relevant concepts** (covered in Learning Notebook Part 2, Part 3, and other materials):
- TF-IDF vectorization and similarity search (the core search problem!)
- Cosine similarity for document retrieval
- LLM integration and prompt engineering
- RAG architecture: Retrieve → Augment → Generate

**💡 Key Insight**: RAG is fundamentally a **search problem**! The retrieval step (finding relevant documents) is the most important part. The LLM just formats the retrieved results into a natural answer.


## What is RAG?

**RAG (Retrieval-Augmented Generation)** combines search with LLM generation:

1. **Retrieval** (The Core Search Problem!): Find relevant information using similarity search
2. **Augmentation**: Include retrieved context in the prompt
3. **Generation**: Use an LLM to format the retrieved results into a natural answer

### Why RAG?

- **LLMs have knowledge cutoff dates**: They may not know about specific movies in our dataset
- **RAG provides up-to-date context**: Uses your actual movie database
- **Better than fine-tuning**: No need to retrain models for domain-specific knowledge
- **Transparency**: You can see what information the LLM used to generate answers

### RAG Flow:

```
User Question: "What are some good sci-fi movies?"
  ↓
RETRIEVAL (Search Problem!): Find top 5 sci-fi movies using TF-IDF similarity search
  ↓
Context: Format movie descriptions into prompt
  ↓
Prompt: "Based on these movie descriptions: [context], answer: What are some good sci-fi movies?"
  ↓
LLM: Format retrieved results into a natural answer
  ↓
Response: "Based on the database, here are some good sci-fi movies: [list]"
```

**💡 Key Insight**: 
- **RAG is fundamentally a search problem!** The retrieval step (finding relevant documents) is the most important part - this is what you've been learning in Parts 1, 2, and 3!
- The LLM just formats the retrieved results into a natural answer. Good retrieval = good RAG results.
- You can also use LLMs as a "judge" to improve, rewrite, or refine search results (see bonus section at the end).


## Setup and Imports

First, let's import all necessary libraries and load the data.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os

# Optional: For LLM integration (uncomment based on your choice)
# import requests  # For Cloudflare Workers AI or other HTTP APIs
# import json
# from transformers import pipeline  # For Hugging Face
# import openai  # For OpenAI API
# import anthropic  # For Anthropic API

print("✅ Libraries imported!")


In [ ]:
# Load movie data
if not os.path.exists('data/movies.csv'):
    print("Data file not found. Downloading from GitHub...")
    os.makedirs('data', exist_ok=True)
    import urllib.request
    url = 'https://raw.githubusercontent.com/samsung-ai-course/8th-9th-edition/main/Chapter%202%20-%20Natural%20Language%20Processing/Class%201%20%26%202%20-%20NLP%20and%20Search/data/movies.csv'
    urllib.request.urlretrieve(url, 'data/movies.csv')
    print("✓ Data file downloaded successfully!")

df = pd.read_csv('data/movies.csv')
print(f"Loaded {len(df)} movies")
df.head()


In [ ]:
# Create TF-IDF vectors for the movie corpus
# This is needed for similarity search (retrieval step)

print("Creating TF-IDF vectors...")
vectorizer = TfidfVectorizer(max_features=100, stop_words='english', lowercase=True)
tfidf_vectors = vectorizer.fit_transform(df['description'])
print(f"✓ Created TF-IDF matrix: {tfidf_vectors.shape}")
print(f"  - {tfidf_vectors.shape[0]} documents (movies)")
print(f"  - {tfidf_vectors.shape[1]} features (words)")


## Step 1: Retrieval Component

**Goal**: Retrieve relevant movies based on the user's question using TF-IDF similarity search.

This is the same similarity search you implemented in Exercise 5 of Part 2. If you haven't completed that exercise, implement it here first!


In [ ]:
def retrieve_relevant_movies(query, vectorizer, tfidf_vectors, df, top_k=5):
    """
    Retrieve relevant movies using TF-IDF similarity search.
    
    This function:
    1. Converts the query to a TF-IDF vector
    2. Calculates cosine similarity with all movie descriptions
    3. Returns the top_k most similar movies
    
    Args:
        query: User question about movies (e.g., "What are good sci-fi movies?")
        vectorizer: Fitted TfidfVectorizer
        tfidf_vectors: TF-IDF matrix for all movie descriptions
        df: DataFrame with movie data
        top_k: Number of movies to retrieve
    
    Returns:
        DataFrame with top_k most relevant movies
    """
    # TODO: Step 1 - Convert query to TF-IDF vector
    # Use vectorizer.transform([query]) - NOT fit_transform! (we already fitted)
    # query_vector = vectorizer.transform([query])
    
    # TODO: Step 2 - Calculate cosine similarity with all documents
    # similarities = cosine_similarity(query_vector, tfidf_vectors)[0]
    
    # TODO: Step 3 - Get indices of top_k most similar documents
    # Hint: Use argsort() to get indices sorted by similarity (descending)
    # top_indices = similarities.argsort()[-top_k:][::-1]
    
    # TODO: Step 4 - Build results DataFrame
    # results = []
    # for idx in top_indices:
    #     results.append({
    #         'movie_id': df.iloc[idx]['movie_id'],
    #         'title': df.iloc[idx]['title'],
    #         'similarity': similarities[idx],
    #         'description': df.iloc[idx]['description']
    #     })
    # return pd.DataFrame(results)
    
    # Placeholder - implement above steps
    return pd.DataFrame()

# Test retrieval
test_query = "What are some good sci-fi movies?"
print(f"Testing retrieval with query: '{test_query}'")
print("=" * 70)

results = retrieve_relevant_movies(test_query, vectorizer, tfidf_vectors, df, top_k=5)

if len(results) > 0:
    print(f"\nRetrieved {len(results)} movies:")
    for idx, row in results.iterrows():
        print(f"\n{idx+1}. {row['title']} (similarity: {row['similarity']:.3f})")
        print(f"   {row['description'][:100]}...")
else:
    print("\n⚠️  No results - implement the function first!")

print("\n💡 Tip: Make sure your retrieval is working before moving to the next step!")


## Step 2: Context Construction

**Goal**: Format the retrieved movies into a context string that will be included in the prompt.

**Key considerations**:
- LLMs have token limits, so keep context concise
- Include relevant information (title, description)
- Format clearly so the LLM can understand it


In [ ]:
def construct_prompt(query, relevant_movies):
    """
    Construct a prompt with retrieved movie context.
    
    This function formats the retrieved movies into a context string
    and combines it with the user's question into a complete prompt.
    
    Args:
        query: User question about movies
        relevant_movies: DataFrame with retrieved movies (from retrieve_relevant_movies)
    
    Returns:
        str: Formatted prompt ready for LLM
    """
    if len(relevant_movies) == 0:
        return f"Question: {query}\n\nNo relevant movies found in the database."
    
    # TODO: Build context from retrieved movies
    # Format example:
    # "Based on the following movie descriptions from our database:\n\n"
    # "Movie 1: [title] - [description]\n"
    # "Movie 2: [title] - [description]\n"
    # "...\n\n"
    # "Question: [query]\n"
    # "Answer:"
    
    context = "Based on the following movie descriptions from our database:\n\n"
    
    # TODO: Loop through relevant_movies and add each movie to context
    # for idx, row in relevant_movies.iterrows():
    #     context += f"Movie {idx+1}: {row['title']} - {row['description']}\n\n"
    
    # TODO: Add the question
    # prompt = context + f"\nQuestion: {query}\n\nAnswer:"
    
    # Placeholder - implement above
    prompt = f"Question: {query}\n\nNo context available (implement function first)."
    
    return prompt

# Test prompt construction
if len(results) > 0:
    prompt = construct_prompt(test_query, results)
    print("Generated Prompt:")
    print("=" * 70)
    print(prompt[:500] + "..." if len(prompt) > 500 else prompt)
    print("\n💡 This prompt will be sent to the LLM!")


## Step 3: LLM Integration

**Choose your LLM provider**: Start with the **Mock LLM** (no setup needed), then try real LLMs!

### Option 0: Mock LLM (Start Here!)

**Perfect for testing your RAG pipeline without any API setup!**

Why start with mock?
- ✅ No setup required
- ✅ Test your retrieval and prompt construction
- ✅ Understand the RAG flow before adding LLM complexity
- ✅ Then upgrade to real LLMs when ready!


In [ ]:
def generate_answer_mock(prompt, relevant_movies):
    """
    Mock LLM for testing RAG pipeline without API access!
    
    This extracts key information from retrieved movies and creates a simple answer.
    Perfect for testing your retrieval and prompt construction!
    
    Args:
        prompt: Full prompt with context (not used in mock, but kept for consistency)
        relevant_movies: DataFrame with retrieved movies
    
    Returns:
        str: Simple answer based on retrieved movies
    """
    if len(relevant_movies) == 0:
        return "I couldn't find any relevant movies in the database."
    
    # TODO: Extract movie titles and create a simple summary
    # This is a simple version - you can make it smarter!
    
    titles = relevant_movies['title'].tolist()
    answer = "Based on the retrieved movies from our database, here are some recommendations:\n\n"
    
    # TODO: Add each movie title with a brief description
    # for i, (idx, row) in enumerate(relevant_movies.iterrows(), 1):
    #     answer += f"{i}. {row['title']}: {row['description'][:100]}...\n"
    
    # Simple version - just list titles
    for i, title in enumerate(titles, 1):
        answer += f"{i}. {title}\n"
    
    return answer

# Test mock LLM
if len(results) > 0:
    mock_answer = generate_answer_mock(prompt, results)
    print("Mock LLM Answer:")
    print("=" * 70)
    print(mock_answer)
    print("\n💡 This is a simple mock - real LLMs will generate more natural answers!")


### Option 1: Cloudflare Workers AI (Recommended for Free Tier)

**Why Cloudflare Workers AI?**
- ✅ Free tier available
- ✅ Easy HTTP API access
- ✅ Good performance

**Setup Steps**:
1. Sign up at https://cloudflare.com (if you don't have an account)
2. Get your Account ID and API Token from Cloudflare dashboard
3. Install requests library: `pip install requests`

**Documentation**: https://developers.cloudflare.com/workers-ai/

**Note**: Cloudflare Workers AI API structure may vary. Check their latest documentation for the current endpoint format.


In [ ]:
def generate_answer_cloudflare(prompt, account_id=None, api_token=None, model="@cf/meta/llama-2-7b-chat-int8"):
    """
    Generate answer using Cloudflare Workers AI.
    
    Args:
        prompt: Full prompt with context
        account_id: Your Cloudflare account ID (optional for some models)
        api_token: Your Cloudflare API token (optional for free tier)
        model: Model name (default: @cf/meta/llama-2-7b-chat-int8)
    
    Returns:
        str: Generated answer
    """
    # Uncomment to use:
    # import requests
    # import json
    
    # TODO: Construct API endpoint
    # Endpoint format: https://api.cloudflare.com/client/v4/accounts/{account_id}/ai/run/{model}
    # Check Cloudflare Workers AI documentation for latest API structure
    
    # TODO: Prepare request
    # url = f"https://api.cloudflare.com/client/v4/accounts/{account_id}/ai/run/{model}"
    # headers = {
    #     "Authorization": f"Bearer {api_token}",  # If needed
    #     "Content-Type": "application/json"
    # }
    # data = {
    #     "messages": [
    #         {"role": "user", "content": prompt}
    #     ]
    # }
    
    # TODO: Make request
    # response = requests.post(url, headers=headers, json=data)
    # result = response.json()
    # return result['result']['response']  # Adjust based on actual API response structure
    
    return "TODO: Implement Cloudflare Workers AI integration. Check documentation for latest API structure."

print("💡 Cloudflare Workers AI integration template provided above.")
print("   Check their documentation: https://developers.cloudflare.com/workers-ai/")


### Option 2: Hugging Face Transformers (Local Inference)

**Why Hugging Face?**
- ✅ Run models locally (no API calls)
- ✅ Free (no API costs)
- ⚠️ Requires GPU for good performance

**Setup**:
```bash
pip install transformers torch
```

**Documentation**: https://huggingface.co/docs/transformers


In [ ]:
def generate_answer_huggingface(prompt, model_name="google/flan-t5-base"):
    """
    Generate answer using Hugging Face Transformers.
    
    Args:
        prompt: Full prompt with context
        model_name: Hugging Face model name
    
    Returns:
        str: Generated answer
    """
    # Uncomment to use:
    # from transformers import pipeline
    
    # TODO: Create text generation pipeline
    # generator = pipeline('text-generation', model=model_name)
    
    # TODO: Generate answer
    # result = generator(prompt, max_length=200, num_return_sequences=1)
    # return result[0]['generated_text']
    
    # Note: For chat models, you might need to use different pipeline types
    # Check Hugging Face documentation: https://huggingface.co/docs/transformers
    
    return "TODO: Implement Hugging Face integration. Uncomment and configure above."

print("💡 Hugging Face integration template provided above.")
print("   Models to try: google/flan-t5-base, microsoft/DialoGPT-medium")
print("   Documentation: https://huggingface.co/docs/transformers")


### Option 3: OpenAI API

**Why OpenAI?**
- ✅ High-quality responses
- ✅ Easy to use
- ⚠️ Requires API key and paid usage

**Setup**:
1. Get API key from https://platform.openai.com/
2. Install: `pip install openai`
3. Set environment variable: `export OPENAI_API_KEY='your-key'`

**Documentation**: https://platform.openai.com/docs


In [ ]:
def generate_answer_openai(prompt, model="gpt-3.5-turbo", api_key=None):
    """
    Generate answer using OpenAI API.
    
    Args:
        prompt: Full prompt with context
        model: OpenAI model name (gpt-3.5-turbo, gpt-4, etc.)
        api_key: OpenAI API key (or set OPENAI_API_KEY environment variable)
    
    Returns:
        str: Generated answer
    """
    # Uncomment to use:
    # import openai
    # import os
    
    # TODO: Set API key
    # if api_key:
    #     openai.api_key = api_key
    # else:
    #     openai.api_key = os.getenv('OPENAI_API_KEY')
    
    # TODO: Make API call
    # response = openai.ChatCompletion.create(
    #     model=model,
    #     messages=[
    #         {"role": "user", "content": prompt}
    #     ],
    #     max_tokens=200
    # )
    # return response.choices[0].message.content
    
    return "TODO: Implement OpenAI integration. Uncomment and configure above."

print("💡 OpenAI integration template provided above.")
print("   Get API key: https://platform.openai.com/")
print("   Documentation: https://platform.openai.com/docs")


## Step 4: Complete RAG Pipeline

**Goal**: Combine all components into a complete RAG system!


In [ ]:
def generate_answer(prompt, llm_provider='mock', relevant_movies=None, **kwargs):
    """
    Generate answer using the specified LLM provider.
    
    Args:
        prompt: Full prompt with context
        llm_provider: 'mock', 'cloudflare', 'huggingface', 'openai', etc.
        relevant_movies: DataFrame with retrieved movies (required for mock)
        **kwargs: Additional arguments for specific LLM providers
    
    Returns:
        str: Generated answer
    """
    if llm_provider == 'mock':
        if relevant_movies is None:
            return "Error: relevant_movies required for mock provider"
        return generate_answer_mock(prompt, relevant_movies)
    
    elif llm_provider == 'cloudflare':
        return generate_answer_cloudflare(prompt, **kwargs)
    
    elif llm_provider == 'huggingface':
        return generate_answer_huggingface(prompt, **kwargs)
    
    elif llm_provider == 'openai':
        return generate_answer_openai(prompt, **kwargs)
    
    else:
        return f"Unknown LLM provider: {llm_provider}"

def rag_query(query, vectorizer, tfidf_vectors, df, top_k=5, llm_provider='mock', **llm_kwargs):
    """
    Complete RAG pipeline: Retrieve → Augment → Generate
    
    Args:
        query: User question about movies
        vectorizer: Fitted TfidfVectorizer
        tfidf_vectors: TF-IDF matrix for all documents
        df: DataFrame with movies
        top_k: Number of movies to retrieve
        llm_provider: LLM provider ('mock', 'cloudflare', 'huggingface', 'openai')
        **llm_kwargs: Additional arguments for LLM (e.g., api_key, model_name)
    
    Returns:
        str: Generated answer
    """
    # Step 1: Retrieve relevant movies
    relevant_movies = retrieve_relevant_movies(query, vectorizer, tfidf_vectors, df, top_k)
    
    # Step 2: Construct prompt with context
    prompt = construct_prompt(query, relevant_movies)
    
    # Step 3: Generate answer using LLM
    answer = generate_answer(prompt, llm_provider, relevant_movies=relevant_movies, **llm_kwargs)
    
    return answer

print("✅ RAG pipeline functions defined!")


## Testing Your RAG System

Test your RAG system with different types of questions!


In [ ]:
# Test queries - try different types of questions!
test_queries = [
    "What are some good sci-fi movies?",
    "Tell me about romantic movies",
    "What movies are about space exploration?",
    "Recommend action movies",
    "What are the best thriller movies?"
]

print("Testing RAG System:")
print("=" * 70)

for query in test_queries[:2]:  # Test first 2 queries
    print(f"\n{'='*70}")
    print(f"Query: {query}")
    print("=" * 70)
    
    answer = rag_query(query, vectorizer, tfidf_vectors, df, top_k=5, llm_provider='mock')
    print(f"\nAnswer:\n{answer}")
    print("\n" + "-" * 70)


## Tips and Best Practices

### 1. Focus on Retrieval (The Search Problem!)

**Remember**: RAG is fundamentally a search problem! The quality of your retrieval determines the quality of your RAG system.

- **Good retrieval** = relevant documents found = good RAG results
- **Bad retrieval** = irrelevant documents = poor RAG results (even with the best LLM!)

**Improve retrieval by**:
- Using better search methods (hybrid search from Part 2)
- Tuning TF-IDF parameters
- Using industry tools from Part 3 (spaCy preprocessing)
- Experimenting with different top_k values

### 2. Prompt Engineering

Experiment with different prompt formats:
- "Based on these movies: [context]. Answer: [query]"
- "Given the following movie descriptions, [query]: [context]"
- "Context: [context]. Question: [query]. Answer:"

### 3. Context Length

LLMs have token limits. Consider:
- Limiting retrieved movies (top_k=3-5)
- Truncating long descriptions
- Summarizing context if needed

### 4. Error Handling

Add error handling for:
- API failures
- Empty retrieval results
- Invalid queries

### 5. Evaluation

Test your RAG system with:
- Genre questions: "What are good action movies?"
- Comparison questions: "Compare sci-fi and fantasy movies"
- Recommendation questions: "Recommend a movie about space"


## Bonus: Using LLM as a Judge to Improve Results

**Advanced Topic** - Optional enhancement!

You can use an LLM not just to generate answers, but also to:
- **Judge/rank** retrieved results for relevance
- **Rewrite/refine** search queries for better retrieval
- **Filter** irrelevant results before generation
- **Summarize** long contexts to fit token limits

### Example: LLM as Query Rewriter

```python
def rewrite_query_with_llm(original_query, llm_provider='mock'):
    """
    Use LLM to rewrite/expand the query for better retrieval.
    Example: "sci-fi movies" → "science fiction movies space exploration futuristic"
    """
    prompt = f"Rewrite this search query to be more specific and include related terms: {original_query}"
    # Use LLM to generate improved query
    improved_query = generate_answer(prompt, llm_provider)
    return improved_query
```

### Example: LLM as Result Judge

```python
def judge_relevance_with_llm(query, retrieved_movies, llm_provider='mock'):
    """
    Use LLM to judge which retrieved movies are most relevant.
    Filter out irrelevant results before generating final answer.
    """
    # Create prompt asking LLM to rank/score relevance
    # Filter movies based on LLM's judgment
    pass
```

**💡 This is advanced!** Start with basic RAG first, then experiment with these enhancements if you're interested.


## Troubleshooting

### Common Issues:

1. **Retrieval returns empty results**
   - Check if your similarity search is implemented correctly
   - Verify TF-IDF vectors are created properly
   - Try different queries

2. **LLM returns errors**
   - Check API keys and authentication
   - Verify API endpoint URLs
   - Check rate limits and quotas

3. **Answers are not relevant**
   - Improve retrieval (try different top_k values)
   - Experiment with prompt formats
   - Check if retrieved movies are actually relevant

4. **Context too long**
   - Reduce top_k (fewer retrieved movies)
   - Truncate descriptions
   - Use shorter prompt format


## Summary

**What you've built**:
- ✅ Complete RAG system for movie Q&A
- ✅ Retrieval component using TF-IDF similarity search (the core search problem!)
- ✅ Context construction and prompt engineering
- ✅ LLM integration (mock and real options)
- ✅ Complete RAG pipeline

**Key Takeaways**:
- **RAG is fundamentally a search problem!** The retrieval step is the most important part.
- Good retrieval = good RAG results. Focus on improving your search methods (from Parts 1, 2, 3).
- The LLM formats retrieved results into natural answers - it's the "presentation layer" on top of search.
- You can use LLMs as judges to improve/rewrite results (bonus/advanced topic).

**What You've Learned**:
- How search (retrieval) is the foundation of RAG
- How to combine search with LLM generation
- How to build a complete RAG pipeline
- How to use different LLM providers

**Next Steps**:
- Experiment with different prompt formats
- Try different LLM providers
- Improve retrieval using techniques from Parts 1, 2, and 3 (hybrid search, better preprocessing, etc.)
- Try the bonus: using LLM as a judge to improve results
- Continue to Class 3 for embeddings and semantic search!

**📚 Resources**:
- Cloudflare Workers AI: https://developers.cloudflare.com/workers-ai/
- Hugging Face: https://huggingface.co/docs/transformers
- OpenAI: https://platform.openai.com/docs
- RAG Guide: https://www.promptingguide.ai/techniques/rag

**💡 Remember**: This is a bonus/take-home exercise. The core learning is in Parts 1, 2, and 3 - RAG just shows how to apply search with LLMs!
